# Agent

An **Agent** runs a loop between a deterministic and a non-deterministic program:

1. Our Python code is a deterministic program which expects structured inputs to function
2. The LLM is a non-deterministic program which can handle natural language and other unstructured inputs

::: {.callout-tip}

To have the two work together, the LLM has to have the minimum function of structuring it's outputs as proper inputs to the Python program.

:::

![Agent Loop](../assets/agent_loop.png)

## Tools

Tools extend what [agents](https://docs.langchain.com/oss/python/langchain/agents) can do—letting them fetch real-time data, execute code, query external databases, and take actions in the world.

Under the hood, tools are callable functions with well-defined inputs and outputs that get passed to a [chat model](https://docs.langchain.com/oss/python/langchain/models). The model decides when to invoke a tool based on the conversation context, and what input arguments to provide.

## Define Internet Search Tool

[**Tavily**](https://www.tavily.com/) is a search engine optimized for LLMs and RAG, aimed at efficient, quick, and persistent search results. As mentioned, it's easy to sign up and offers a generous free tier.

```sh
!pip install tavily-python
```

#### Set up Tavily API for web search (Free)

* Tavily Search API is a search engine optimized for LLMs and RAG, aimed at efficient, 
quick, and persistent search results. 
* You can sign up for an API key [here](https://tavily.com/). 
It's easy to sign up and offers a very generous free tier. Some lessons (in Module 4) will use Tavily. 

* Add `TAVILY_API_KEY` to Colab Secrets (key icon in the left sidebar).

In [ ]:
from google.colab import userdata
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=userdata.get("TAVILY_API_KEY"))

In [ ]:
from typing import Literal


def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

In [ ]:
question = "What's the timeline of how the AI industry has evolved over the past 10 years?"
result = internet_search(question, max_results=3)
result

{'query': 'What is LangGraph?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.ibm.com/think/topics/langgraph',
   'title': 'What is LangGraph? - IBM',
   'content': 'LangGraph, created by LangChain, is an open source AI agent framework designed to build, deploy and manage complex generative AI agent workflows. At its core, LangGraph uses the power of graph-based architectures to model and manage the intricate relationships between various components of an AI agent workflow. LangGraph illuminates the processes within an AI workflow, allowing full transparency of the agent’s state. By combining these technologies with a set of APIs and tools, LangGraph provides users with a versatile platform for developing AI solutions and workflows including chatbots, state graphs and other agent-based systems. **Nodes**: In LangGraph, nodes represent individual components or agents within an AI workflow. LangGraph uses enhanced decision-making by model

## Create Agent (with tools)

The [`create_agent`](https://reference.langchain.com/python/langchain/agents/factory/create_agent) factory function provides a production-ready **agent implementation**. It creates an agent graph that calls tools in a loop until a stopping condition is met.

[View source code on GitHub](https://github.com/langchain-ai/langchain/blob/fb6ab993a73180538f6cca876b3c85d46c08845f/libs/langchain_v1/langchain/agents/factory.py#L691).

### 1. Model: the non-deterministic program

Since agents require [**a model that supports tool calling**](https://openrouter.ai/models?fmt=cards&supported_parameters=tools). Filtering **Tool Calling** models that are also **Free**, we select NVIDIA's Nemotron-3:

In [ ]:
import os

In [ ]:
from langchain_openai import ChatOpenAI

# https://openrouter.ai/nvidia/nemotron-3-nano-30b-a3b:free
model_nemotron3_nano = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    # OpenRouter instead of the default OpenAI API
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

### 2. System prompt: natural language instructions

- LLMs were tuned to prioritize **System prompt** over user messages.
    - **prompt injection attack** happens when they don't.
- LLMs have been trained on many styles of instruction to do various text output generation.

In [ ]:
from textwrap import dedent


AGENT_PROMPT = dedent("""
    Role: You are an expert researcher. Your job is to conduct thorough research and then write a polished report.
    Style: Keep it short and concise.
    Tools: You have access to an `internet_search` tool as your primary means of gathering information.

    ## Tool: `internet_search`

    - Usage: Use this to run an internet search for a given query.
    - Parameters: You can specify the max number of results to return, the topic, and whether raw content should be included.
""")

See [Prompt engineering concepts](https://docs.langchain.com/langsmith/prompt-engineering-concepts) for more details.

### 3. The `agent` class instance

Many models have been trained to **format their output in JSON** and stop immediatly. This is done for parsability by deterministic programs like Python code:

1. Output stops after generating JSON
2. Python (`agent` class instance) parses this output
3. It indicates calling a specific function with specific arguments
4. Python calls the function and passess the arguments and assigns it to a variable
5. Check for stop condition
6. Python append the value as a string to the prompt
7. Python re-triggeres the LLM with this result
8. .. repeat (loop)

![Figure: Function Calling Steps | OpenAI](https://cdn.openai.com/API/docs/images/function-calling-diagram-steps.png){height=600}

See: [Guide > Function Calling | OpenAI](https://developers.openai.com/api/docs/guides/function-calling).

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model_nemotron3_nano, # Program (non-deterministic)
    tools=[internet_search],    # I/O (deterministic)
    system_prompt=AGENT_PROMPT  # Instructions (natural language)
)

## Invoke the `agent`

In [ ]:
from langchain_core.messages import HumanMessage


result = agent.invoke({
    "messages": [
        HumanMessage(question)
    ]
})

In [123]:
result

{'messages': [HumanMessage(content='Explain agentic AI in a tweet', additional_kwargs={}, response_metadata={}, id='eb3b86d8-3111-4646-bd11-7f2b3d24e2cb'),
  AIMessage(content='Agentic AI refers to AI systems that can autonomously set goals, plan, and act to achieve them without constant human input—think of AI agents that decide, adapt, and execute tasks like a digital personal assistant on steroids.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 916, 'prompt_tokens': 456, 'total_tokens': 1372, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 651, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-nano-30b-a3b:free', 

In [124]:
# Print the agent's response
print(result["messages"][-1].content)

Agentic AI refers to AI systems that can autonomously set goals, plan, and act to achieve them without constant human input—think of AI agents that decide, adapt, and execute tasks like a digital personal assistant on steroids.


Let's ask about something that needs search:

In [ ]:
result = agent.invoke({
    "messages": [
        HumanMessage("What's the difference between a Model and an Agent?")
    ]
})

In [128]:
result

{'messages': [HumanMessage(content='What is the difference between LangChain, LangGraph and LangSmith?', additional_kwargs={}, response_metadata={}, id='7f8c9005-eb8e-4072-86e0-64dd26bec6a6'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 123, 'prompt_tokens': 462, 'total_tokens': 585, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 66, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-nano-30b-a3b:free', 'system_fingerprint': None, 'id': 'gen-1771856086-8OPWhAITZng1xT0GmOTD', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c8ada-45c5-7e13-8c63-b2bdba91d977-0', tool_calls=[{'na

In [126]:
type(result["messages"][-2])

langchain_core.messages.tool.ToolMessage

In [127]:
# Print the agent's response
print(result["messages"][-1].content)

**LangChain vs LangGraph vs LangSmith – What They Do and When to Use Them**

| Aspect | **LangChain** | **LangGraph** | **LangSmith** |
|--------|---------------|---------------|---------------|
| **Primary Goal** | Provide a **modular toolbox** for building LLM‑powered apps (prompts, models, memory, tools, RAG, etc.). | Add **stateful, graph‑based orchestration** for complex, multi‑step workflows that need branching, loops, retries, and shared state. | Offer **observability, tracing, evaluation, and monitoring** for any LLM application (especially those built with LangChain/LangGraph). |
| **Core Concept** | **Chains** – linear pipelines of components (`prompt → model → parser`). | **Graphs** – nodes (steps/agents) and edges (control flow) that can loop, branch, and retain a mutable state. | **Traces** – full execution logs that let you drill from a workflow run down to a single LLM call, plus evaluation & metrics. |
| **Typical Use‑Cases** | • Simple chatbots, Q&A, RAG pipelines<br>•

## Toolkits

A **toolkit** is a collection of tools meant to be used together.

Each tools within a toolkit is designed to be called by a model:

- their inputs are designed to be generated by models
- their outputs are designed to be passed back to models

See [Tool integrations](https://docs.langchain.com/oss/python/integrations/tools) which cover ready-made tools for various things like:

- [Search](https://docs.langchain.com/oss/python/integrations/tools#search)
- [Productivity](https://docs.langchain.com/oss/python/integrations/tools#productivity)
- [Web browsing](https://docs.langchain.com/oss/python/integrations/tools#web-browsing)
- [Database](https://docs.langchain.com/oss/python/integrations/tools#database)
- [Code interpreter](https://docs.langchain.com/oss/python/integrations/tools#code-interpreter)
- [others](https://docs.langchain.com/oss/python/integrations/tools#all-tools-and-toolkits)

## MCP Servers

## Skills

[Agent -> Skills -> MCP Server -> Tools](https://youtu.be/mYSRn6PC1mc?si=hMLxU49UhM6HsHkQ&t=3655).
**Putting it together:** When a user submits a query, the agent identifies the relevant **Skill**. This skill directs the agent to use specific **Tools** exposed by the **MCP Server**. The agent orchestrates these tools, processes the data, and iterates until the research goal is met, ensuring a reliable and modular end-to-end workflow (1:04:30-1:05:43).


## Key Takeaways

- Agent = Model + Tools
- Models (LLMs) are the brain-power of agents
- Tools connect the LLM to the outside world.